# 06 · Score the training set

Gradients for the chosen mixture's balanced scoring set, cached once, then
scored against four trait directions, 96 null directions, and two non-direction
baselines.

**Scoring set.** All A examples plus an equal random sample of N. At a 10% dose
that is 1,000 + 1,000 rather than 10,000: the other 8,000 negatives add nothing
to the negative distribution and cost eight times the GPU. Balancing also makes
P@k and average precision comparable across fractions, which they are not on the
raw mixtures.

**Scoring model is the base.** `attribution.scoring_model: base` — that is what
a real auditor holds, and taking gradients under the student would leak the
answer. `assert_no_adapter` enforces it.

> Every number here is **provisional** until notebook 07 shows the same
> pipeline returns chance on placebo labels.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib, subprocess
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

# WHICH code is running? Nothing else in this notebook would notice a `main`
# checkout until a missing file several cells in, and pivot and main answer
# different questions -- their results have to stay independently attributable.
# sys.path puts ROOT/src first so the working tree beats any installed copy;
# assert that rather than assume it.
BRANCH = subprocess.run(
    ["git", "-C", str(ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()
assert BRANCH == "pivot", f"expected the 'pivot' branch at {ROOT}, found {BRANCH!r}"
assert Path(config.__file__).resolve().is_relative_to(ROOT / "src"), (
    f"subattr is imported from {config.__file__}, not {ROOT / 'src'}"
)
assert config.REPO_ROOT == ROOT, f"REPO_ROOT is {config.REPO_ROOT}, not {ROOT}"

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"branch    {BRANCH}   {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
# The fraction chosen by the notebook-02 gate. Every stage after 02 reads it
# rather than hard-coding a dose, so the whole pipeline moves together if the
# gate is ever re-run.
GATE = json.loads((RUN / "gate.json").read_text())
FRACTION = GATE["fraction"]
print(f"gate fraction: {FRACTION}   (rule: {GATE['rule']})")

In [ ]:
from subattr import attribution as A
from subattr import baselines as bl
from subattr import ingest as ing
from subattr import metrics as M
from subattr import mixtures as mx
from subattr import train as tr
from subattr.cache import free_gpu, gpu_memory, load_tensors

import pandas as pd
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

deltas = load_tensors(RUN / "deltas.pt")
nulls = load_tensors(RUN / "nulls.pt")
LAYER = json.loads((RUN / "preregistered_layer.json").read_text())["layer"]
HEADLINE = ["delta_iso", "delta_mixed", "delta_clean", "delta_pureA"]
print(f"{len(deltas)} trait directions, {len(nulls)} null directions, layer {LAYER}")

In [ ]:
rows = ing.read_jsonl(MIX / f"{FRACTION}_mixed.jsonl")
sources = [r["source"] for r in ing.read_jsonl(MIX / f"{FRACTION}_provenance.jsonl")]

RULE = "every A example, plus an equal-sized random sample of N, seed=cfg.seed"
subset = mx.balanced_subset(sources, positive="A", seed=cfg.seed)
examples = [rows[i] for i in subset]
labels = [int(sources[i] == "A") for i in subset]
print(f"scoring set rule : {RULE}")
print(f"scoring set      : {len(examples)} examples, {sum(labels)} A / "
      f"{len(labels) - sum(labels)} N, from {FRACTION}_mixed.jsonl")

(RUN / "scoring_set.json").write_text(json.dumps({
    "fraction": FRACTION, "rule": RULE, "seed": cfg.seed,
    "mixture_indices": subset, "n": len(subset), "n_pos": int(sum(labels)),
}, indent=2))

GRADCACHE = RUN / f"gradcache_{FRACTION}"
LOSS_NAME = f"loss_student_{FRACTION}.pt"
SCORES_NAME = f"scores_{FRACTION}.parquet"
NULL_NAME = f"null_{FRACTION}.parquet"
TABLE_NAME = f"table_{FRACTION}.csv"
FIG_NAME = f"fig_auroc_{FRACTION}.png"
STUDENT = FRACTION
TITLE = f"{FRACTION} training set"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)
base = AutoModelForCausalLM.from_pretrained(
    cfg.base_model, dtype=torch.bfloat16, device_map="auto"
).eval()
A.assert_no_adapter(base)   # gradients must be taken under the BASE model

A.cache_gradient_features(
    base, tokenizer, examples, GRADCACHE,
    chunk_size=250, token_grad_layer=LAYER, progress_every=100,
)
features = A.load_gradient_features(GRADCACHE)
print(f"cached {features['sum_response'].shape[0]} examples, "
      f"{features['sum_response'].shape[1]} layers")
print(f"[gpu] {gpu_memory()}")

In [ ]:
# The cache is only valid if mean_response really is sum / n_scored.
probe = A.score_from_cache(
    {k: v[:64] for k, v in features.items()}, {"delta_iso": deltas["delta_iso"]},
    aggregations=("sum_response", "mean_response"),
)
wide = probe.pivot_table(index=["example_index", "layer"], columns="aggregation", values="score")
n_scored = features["n_scored"][:64].float().numpy()
expected = n_scored.repeat(features["sum_response"].shape[1])
total = wide["sum_response"].to_numpy()
gap = abs(total - wide["mean_response"].to_numpy() * expected)
assert (gap / pd.Series(abs(total)).clip(lower=1e-6).to_numpy()).max() < 1e-3, (
    "mean_response is not sum_response / n_scored"
)
print("OK: mean_response == sum_response / n_scored")
print(f"scored tokens per example: median {int(pd.Series(n_scored).median())}, "
      f"range [{int(n_scored.min())}, {int(n_scored.max())}]")

In [ ]:
# L_base comes free from the gradient cache; only L_student needs a pass.
loss_base = features["loss"]
loss_path = RUN / LOSS_NAME
if loss_path.exists():
    loss_student = torch.load(loss_path, map_location="cpu", weights_only=True)
    print(f"[cache] student losses: loaded from {loss_path}")
else:
    student = PeftModel.from_pretrained(base, tr.latest_adapter(str(RUN / "students" / STUDENT)))
    student.eval()
    try:
        loss_student = bl.response_losses(student, tokenizer, examples)
    finally:
        base = student.unload()
    torch.save(loss_student, loss_path)
    print(f"[cache] student losses: saved to {loss_path}")

print(f"mean L_base {float(loss_base.mean()):.4f}   "
      f"mean L_student({STUDENT}) {float(loss_student.mean()):.4f}")
free_gpu(base, tokenizer)
print(f"[gpu] {gpu_memory()}")

In [ ]:
scores = A.score_from_cache(features, deltas)
scores = pd.concat([scores, bl.grad_norm_frame(features),
                    bl.loss_gap_frame(loss_base, loss_student)], ignore_index=True)
scores.to_parquet(RUN / SCORES_NAME, index=False)
print(f"{len(scores):,} rows -> {RUN / SCORES_NAME}")
print(sorted(scores.direction.unique()))

In [ ]:
# The null stays wide: melting 96 directions x 29 layers x 4 aggregations to
# long form is >100M rows, and every one of them is reduced to an AUROC anyway.
null_frames = []
names = list(nulls)
for start in range(0, len(names), 8):
    group = {k: nulls[k] for k in names[start : start + 8]}
    null_frames.append(M.auroc_grid(A.score_tensors(features, group), labels))
    print(f"  scored nulls {min(start + 8, len(names))}/{len(names)}", flush=True)
null = pd.concat(null_frames, ignore_index=True)
null.to_parquet(RUN / NULL_NAME, index=False)
print(f"{len(null):,} null cells -> {RUN / NULL_NAME}")

In [ ]:
table = M.scorer_table(
    scores, labels, k=int(sum(labels)), n_boot=1000, seed=cfg.seed,
    bootstrap_layers=[LAYER, -1], null=null,
)
table.to_csv(RUN / TABLE_NAME, index=False)

cols = ["direction", "aggregation", "auroc", "auroc_lo", "auroc_hi", "ap", "p_at_k",
        "null_random_p95", "null_random_pct", "null_random_p",
        "null_covrand_p95", "null_covrand_pct", "null_covrand_p"]
headline = table[table.layer.isin([LAYER, -1])].copy()
headline["order"] = headline.direction.map(
    {d: i for i, d in enumerate(HEADLINE + ["loss_gap", "grad_norm"])}
).fillna(99)
headline = headline.sort_values(["order", "aggregation"])
print(f"=== layer {LAYER} (pre-registered) + layer-free baselines ===")
print(headline[cols].to_string(index=False, float_format=lambda v: f"{v:7.4f}"))

In [ ]:
import matplotlib.pyplot as plt

grid = table[(table.layer >= 0) & (table.aggregation == "sum_response")]
pivot = grid.pivot(index="direction", columns="layer", values="auroc")
pivot = pivot.reindex([d for d in HEADLINE + ["grad_norm"] if d in pivot.index])

fig, ax = plt.subplots(figsize=(11, 2.6))
im = ax.imshow(pivot.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=0.3, vmax=0.7)
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_xticks(range(0, pivot.shape[1], 2), pivot.columns[::2])
ax.set_xlabel("residual slot (0 = embedding)")
ax.axvline(LAYER, color="k", lw=1.2, ls="--")
ax.set_title(f"AUROC, sum_response, all layers -- {TITLE}  (dashed = pre-registered layer)")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(RUN / FIG_NAME, dpi=140)
plt.show()

print("\nExploratory: this is a maximum over 29 correlated layers. Only the "
      "dashed column is a pre-registered number.")

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.